[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/05.%20The%20Single%20Truck%20Gate%20Entry%20Analysis/P5-Tier-4_executed.ipynb) [![Open In SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 5. The Single Truck Gate Entry Analysis

## Tier 4 — A Deep Q-Network for Gate Assignment

### Goal
Build and train a Deep Q-Network (DQN) agent that assigns each arriving truck to one of several gates. The agent observes the current operating state, chooses a gate, and receives a reward based on the truck's waiting time and priority. The notebook demonstrates the complete learning workflow: environment design, neural-network approximation, experience replay, target-network updates, training, evaluation, and visualization.

### Inputs and outputs
- **Inputs:** the number of gates, the number of trucks per episode, the arrival-time horizon, and the DQN hyperparameters.
- **Agent input:** a numeric state vector describing gate availability relative to the current truck, that truck's priority and service time, the remaining queue length, and its arrival time.
- **Agent output:** one discrete action, namely the index of the gate assigned to the current truck.
- **Reported outputs:** episode rewards, training losses, and evaluation averages for reward, waiting time, throughput, and gate utilization.

### Modeling assumptions and scope
- Truck arrival times are sampled uniformly over the simulation duration. Service times are sampled uniformly from 3 to 12 minutes, and priorities are sampled from the specified weighted distribution.
- Trucks are considered in nondecreasing arrival-time order. Each truck is assigned immediately to a gate; its service starts at the later of its arrival and that gate's availability.
- The simulation duration is the arrival-time horizon, not a deadline for completing service. An episode processes every generated truck, including service that finishes after the arrival horizon.
- The state summarizes the current decision and gate availability; it does not expose the full future truck schedule. The agent therefore learns from the stochastic arrival and truck attributes represented by the environment rather than being given a complete schedule.
- The model does not include gate-specific service rates, travel distances, vehicle compatibility, operating costs, or constraints beyond assigning each truck to one gate. DQN training does not guarantee a globally optimal policy or generalization to other operating conditions.

### Learning method
At each decision, the agent may explore by choosing a random gate or exploit its current estimates by choosing the gate with the largest estimated action value. The DQN learns these values from sampled transitions stored in a replay buffer. A separate target network supplies the learning targets and is periodically synchronized with the online network.

For a state \(s_t\), gate action \(a_t\), reward \(r_t\), next state \(s_{t+1}\), and terminal indicator \(d_t\), the one-step Q-learning target is

$$
y_t = r_t + \gamma (1-d_t) \max_{a'} Q_{\text{target}}(s_{t+1}, a').
$$

For a replay batch of size \(B\), the online network is trained to minimize

$$
L = \frac{1}{B} \sum_{t=1}^{B} \left(Q_{\text{online}}(s_t,a_t)-y_t\right)^2.
$$

Here, \(Q(s,a)\) estimates the discounted future reward from taking action \(a\) in state \(s\), and \(\gamma\) is the discount factor. The terminal indicator prevents the target from including future value after the final truck. In the code, these calculations appear in `DQNAgent.replay`; the neural-network architecture is defined by `DQN`.

### What to examine
- The reward curve shows how the total reward varies across training episodes; a moving average helps reveal the broader trend.
- The loss curve tracks the squared error between the online network's estimates and the Q-learning targets. A lower loss alone does not establish good operational performance.
- The epsilon curve records the exploration rate at the end of each episode. Exploration decreases during replay updates, subject to the configured minimum.
- Evaluation metrics summarize behavior on newly generated episodes with exploration disabled. They are empirical results for this run, not guaranteed outcomes or comparisons with another method.

In [ ]:
# Import the libraries used for simulation, neural-network training, and plots.
import random
import warnings
from collections import deque

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class GateEnvironment:
    """Simulation environment for sequential truck-to-gate assignments."""

    def __init__(self, num_gates=3, max_trucks=50, simulation_duration=100):
        if num_gates <= 0:
            raise ValueError("num_gates must be positive.")
        if max_trucks <= 0:
            raise ValueError("max_trucks must be positive.")
        if simulation_duration <= 0:
            raise ValueError("simulation_duration must be positive.")

        self.num_gates = num_gates
        self.max_trucks = max_trucks
        self.simulation_duration = simulation_duration

        # The state contains one relative availability value per gate and four truck/queue features.
        self.action_space = num_gates
        self.observation_space = num_gates + 4

        self.reset()

    def reset(self):
        """Start a new episode and return its initial observation."""
        self.current_time = 0.0
        self.trucks_processed = 0
        self.total_waiting_time = 0.0

        self.gate_available_time = np.zeros(self.num_gates, dtype=float)
        self.gate_utilization = np.zeros(self.num_gates, dtype=float)

        self.truck_queue = self._generate_truck_queue()
        self.current_truck_index = 0

        return self._get_state()

    def _generate_truck_queue(self):
        """Sample truck attributes and order trucks by arrival time."""
        trucks = []

        for i in range(self.max_trucks):
            arrival_time = random.uniform(0, self.simulation_duration)
            service_time = random.uniform(3, 12)
            priority = random.choices(
                [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
                weights=[5, 8, 12, 15, 18, 15, 12, 8, 5, 2],
            )[0]

            trucks.append(
                {
                    "id": i + 1,
                    "arrival_time": arrival_time,
                    "service_time": service_time,
                    "priority": priority,
                }
            )

        return sorted(trucks, key=lambda truck: truck["arrival_time"])

    def _get_state(self):
        """Return the feature vector for the next truck and current gate state."""
        if self.current_truck_index >= len(self.truck_queue):
            return np.zeros(self.observation_space, dtype=float)

        current_truck = self.truck_queue[self.current_truck_index]
        arrival_time = current_truck["arrival_time"]

        # Signed relative times preserve both busy gates and gates already idle at arrival.
        relative_gate_availability = (
            self.gate_available_time - arrival_time
        ) / self.simulation_duration
        current_priority = current_truck["priority"] / 10.0
        service_time = current_truck["service_time"] / 12.0
        queue_length = (
            len(self.truck_queue) - self.current_truck_index
        ) / self.max_trucks
        arrival_time_ratio = arrival_time / self.simulation_duration

        return np.concatenate(
            [
                relative_gate_availability,
                [
                    current_priority,
                    service_time,
                    queue_length,
                    arrival_time_ratio,
                ],
            ]
        )

    def step(self, action):
        """Assign the current truck and return state, reward, done, and info."""
        if not isinstance(action, (int, np.integer)):
            raise TypeError("action must be an integer gate index.")
        if not 0 <= action < self.num_gates:
            raise ValueError("action must identify an existing gate.")

        if self.current_truck_index >= len(self.truck_queue):
            return self._get_state(), 0.0, True, {}

        current_truck = self.truck_queue[self.current_truck_index]
        gate_id = int(action)

        start_time = max(
            current_truck["arrival_time"], self.gate_available_time[gate_id]
        )
        completion_time = start_time + current_truck["service_time"]
        waiting_time = start_time - current_truck["arrival_time"]

        self.gate_available_time[gate_id] = completion_time
        self.gate_utilization[gate_id] += current_truck["service_time"]

        self.total_waiting_time += waiting_time
        self.trucks_processed += 1
        self.current_time = max(self.current_time, completion_time)
        self.current_truck_index += 1

        reward = self._calculate_reward(
            waiting_time, current_truck["priority"]
        )
        done = self.current_truck_index >= len(self.truck_queue)

        info = {
            "waiting_time": waiting_time,
            "truck_id": current_truck["id"],
            "gate_id": gate_id,
            "cumulative_waiting": self.total_waiting_time,
        }

        return self._get_state(), reward, done, info

    def _calculate_reward(self, waiting_time, priority):
        """Reward low waiting time, with the specified priority bonus."""
        waiting_penalty = -waiting_time * 0.1
        priority_bonus = priority * 0.05
        return waiting_penalty + priority_bonus

    def get_performance_metrics(self):
        """Calculate average waiting time, throughput, and gate utilization."""
        if self.trucks_processed == 0:
            return {"avg_waiting": 0.0, "throughput": 0.0, "utilization": 0.0}

        avg_waiting = self.total_waiting_time / self.trucks_processed
        throughput = (
            self.trucks_processed / (self.current_time / 60)
            if self.current_time > 0
            else 0.0
        )
        total_utilization = np.sum(self.gate_utilization)
        avg_utilization = (
            total_utilization / (self.current_time * self.num_gates)
            if self.current_time > 0
            else 0.0
        )

        return {
            "avg_waiting": avg_waiting,
            "throughput": throughput,
            "utilization": avg_utilization,
        }

## DQN architecture and training procedure

### State, action, and reward
The environment presents a state vector with one relative gate-availability feature per gate, followed by the current truck's normalized priority, normalized service time, normalized remaining queue length, and normalized arrival time. A negative relative availability value means that the gate is idle at the current truck's arrival; a positive value indicates how long after arrival the gate becomes available. Each action selects one gate. The reward is

$$
r_t = -0.1 w_t + 0.05 p_t,
$$

where \(w_t\) is the truck's waiting time in minutes and \(p_t\) is its priority on the 1-to-10 scale. The priority term is constant across gate choices for a given truck; the gate-dependent part of this reward is the waiting-time penalty.

### Epsilon-greedy action selection
Exploration lets the agent collect experience from different gate choices; exploitation uses its current value estimates. With probability \(\epsilon\), `DQNAgent.act` selects a random gate. Otherwise, it selects the gate with the largest online-network Q-value. During evaluation, exploration is disabled.

### Experience replay and target network
The replay buffer stores transitions \((s_t,a_t,r_t,s_{t+1},d_t)\). Randomly sampling a mini-batch reduces dependence on the order in which consecutive transitions were generated. The target network supplies the next-state values in the Bellman target and is copied from the online network at the configured update interval. These mechanisms are part of the DQN training method implemented below; they do not guarantee convergence for every environment or parameter choice.

The network has two hidden layers with 64 and 32 ReLU units. Its output contains one estimated Q-value for each gate. The optimizer minimizes the mean squared Bellman-target error over replay batches.

In [ ]:
class DQN(nn.Module):
    """Feed-forward neural network that estimates one Q-value per action."""

    def __init__(self, state_size, action_size, hidden_sizes=(64, 32)):
        super().__init__()

        layers = []
        layers.append(nn.Linear(state_size, hidden_sizes[0]))
        layers.append(nn.ReLU())

        for index in range(len(hidden_sizes) - 1):
            layers.append(nn.Linear(hidden_sizes[index], hidden_sizes[index + 1]))
            layers.append(nn.ReLU())

        layers.append(nn.Linear(hidden_sizes[-1], action_size))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


class DQNAgent:
    """DQN agent with experience replay and a periodically updated target network."""

    def __init__(self, state_size, action_size, lr=0.001):
        self.state_size = state_size
        self.action_size = action_size
        self.lr = lr

        self.q_network = DQN(state_size, action_size)
        self.target_network = DQN(state_size, action_size)
        self.target_network.load_state_dict(self.q_network.state_dict())

        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        self.memory = deque(maxlen=10000)

        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.1
        self.epsilon_decay = 0.995
        self.batch_size = 32
        self.target_update_freq = 100

        self.training_step = 0
        self.losses = []
        self.epsilon_history = []

    def remember(self, state, action, reward, next_state, done):
        """Store one transition so it can be reused in later mini-batches."""
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state, training=True):
        """Choose a random exploratory action or the highest-valued action."""
        if training and random.random() <= self.epsilon:
            return random.randrange(self.action_size)

        state_tensor = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            q_values = self.q_network(state_tensor)

        return int(torch.argmax(q_values, dim=1).item())

    def replay(self):
        """Update the online network using a random replay-buffer mini-batch."""
        if len(self.memory) < self.batch_size:
            return None

        batch = random.sample(self.memory, self.batch_size)
        states = torch.as_tensor(
            np.asarray([experience[0] for experience in batch]),
            dtype=torch.float32,
        )
        actions = torch.as_tensor(
            [experience[1] for experience in batch], dtype=torch.long
        )
        rewards = torch.as_tensor(
            [experience[2] for experience in batch], dtype=torch.float32
        )
        next_states = torch.as_tensor(
            np.asarray([experience[3] for experience in batch]),
            dtype=torch.float32,
        )
        dones = torch.as_tensor(
            [experience[4] for experience in batch], dtype=torch.float32
        )

        current_q_values = self.q_network(states).gather(
            1, actions.unsqueeze(1)
        ).squeeze(1)

        with torch.no_grad():
            next_q_values = self.target_network(next_states).max(dim=1).values
            target_q_values = rewards + self.gamma * next_q_values * (1 - dones)

        loss = F.mse_loss(current_q_values, target_q_values)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.losses.append(loss.item())

        if self.epsilon > self.epsilon_min:
            self.epsilon = max(
                self.epsilon_min, self.epsilon * self.epsilon_decay
            )

        self.training_step += 1
        if self.training_step % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())

        return loss.item()

    def train(self, env, num_episodes=1000):
        """Train the agent and return reward and mean-loss histories."""
        if num_episodes <= 0:
            raise ValueError("num_episodes must be positive.")

        episode_rewards = []
        episode_losses = []
        self.epsilon_history = []

        print(f"Starting DQN training for {num_episodes} episodes...")
        print("=" * 60)

        for episode in range(num_episodes):
            state = env.reset()
            total_reward = 0.0
            losses_before_episode = len(self.losses)

            while True:
                action = self.act(state)
                next_state, reward, done, _ = env.step(action)
                self.remember(state, action, reward, next_state, done)

                state = next_state
                total_reward += reward
                self.replay()

                if done:
                    break

            episode_rewards.append(total_reward)
            new_losses = self.losses[losses_before_episode:]
            episode_losses.append(
                float(np.mean(new_losses)) if new_losses else float("nan")
            )
            self.epsilon_history.append(self.epsilon)

            if (episode + 1) % 100 == 0:
                recent_rewards = np.mean(episode_rewards[-100:])
                recent_losses = np.asarray(episode_losses[-100:], dtype=float)
                finite_losses = recent_losses[np.isfinite(recent_losses)]
                average_loss = (
                    float(np.mean(finite_losses)) if finite_losses.size else float("nan")
                )
                print(
                    f"Episode {episode + 1:4d}: "
                    f"Average reward = {recent_rewards:.3f}, "
                    f"epsilon = {self.epsilon:.3f}, "
                    f"mean episode loss = {average_loss:.6f}"
                )

        print(f"\nTraining completed. Final epsilon: {self.epsilon:.3f}")
        return episode_rewards, episode_losses

    def evaluate(self, env, num_episodes=10):
        """Evaluate the greedy policy on newly generated episodes."""
        if num_episodes <= 0:
            raise ValueError("num_episodes must be positive.")

        print(f"\nEvaluating the agent for {num_episodes} episodes...")
        print("=" * 50)

        total_rewards = []
        performance_metrics = []

        for _ in range(num_episodes):
            state = env.reset()
            total_reward = 0.0

            while True:
                action = self.act(state, training=False)
                next_state, reward, done, _ = env.step(action)
                state = next_state
                total_reward += reward

                if done:
                    break

            total_rewards.append(total_reward)
            performance_metrics.append(env.get_performance_metrics())

        avg_reward = float(np.mean(total_rewards))
        avg_metrics = {
            key: float(np.mean([metrics[key] for metrics in performance_metrics]))
            for key in performance_metrics[0]
        }

        print(f"Evaluation average reward: {avg_reward:.3f}")
        print(f"Average waiting time: {avg_metrics['avg_waiting']:.2f} minutes")
        print(f"Throughput: {avg_metrics['throughput']:.1f} trucks/hour")
        print(f"Gate utilization: {avg_metrics['utilization'] * 100:.1f}%")

        return avg_reward, avg_metrics

## Train and evaluate the agent

The following cell creates the specified three-gate environment, trains the DQN for 1,000 episodes, and evaluates the greedy policy on 10 newly generated episodes. The evaluation episodes are stochastic, so their metrics depend on the sampled trucks and the learned network. Training is not evidence by itself that the policy is optimal; the reward and operational metrics should be interpreted together.

In [ ]:
print("DQN reinforcement learning for gate assignment")
print("=" * 70)
print("Training a neural-network policy in the gate assignment environment...")

env = GateEnvironment(
    num_gates=3,
    max_trucks=50,
    simulation_duration=100,
)

agent = DQNAgent(
    state_size=env.observation_space,
    action_size=env.action_space,
    lr=0.001,
)

episode_rewards, episode_losses = agent.train(env, num_episodes=1000)
avg_reward, avg_metrics = agent.evaluate(env, num_episodes=10)

print("\nEvaluation summary")
print(f"Average reward: {avg_reward:.3f}")
print(f"Average waiting time: {avg_metrics['avg_waiting']:.2f} minutes")
print(f"System throughput: {avg_metrics['throughput']:.1f} trucks/hour")
print(f"Gate utilization: {avg_metrics['utilization'] * 100:.1f}%")

## Visualize training and evaluation

The reward plot includes both episode-level observations and a moving average. Losses are averaged over the replay updates performed in each episode; episodes before the replay buffer contains a full mini-batch have no loss value. The epsilon plot uses the rate recorded at the end of each episode rather than reconstructing a hypothetical schedule. The summary panel reports measurements from the training and evaluation run.

In [ ]:
if episode_rewards:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("DQN Gate-Assignment Training", fontsize=16, fontweight="bold")

    episodes = np.arange(1, len(episode_rewards) + 1)

    reward_axis = axes[0, 0]
    window_size = 50
    moving_average = [
        np.mean(episode_rewards[max(0, index - window_size + 1): index + 1])
        for index in range(len(episode_rewards))
    ]
    reward_axis.plot(
        episodes,
        moving_average,
        color="blue",
        linewidth=2,
        label=f"Moving average ({window_size} episodes)",
    )
    reward_axis.plot(
        episodes,
        episode_rewards,
        color="gray",
        alpha=0.3,
        label="Episode reward",
    )
    reward_axis.set_title("Reward by Episode")
    reward_axis.set_xlabel("Episode")
    reward_axis.set_ylabel("Total reward")
    reward_axis.grid(True, alpha=0.3)
    reward_axis.legend()

    loss_axis = axes[0, 1]
    loss_values = np.asarray(episode_losses, dtype=float)
    valid_losses = np.isfinite(loss_values)
    if np.any(valid_losses):
        loss_axis.plot(
            episodes[valid_losses],
            loss_values[valid_losses],
            color="red",
            linewidth=1.5,
        )
        loss_axis.set_yscale("log")
    loss_axis.set_title("Mean Replay Loss by Episode")
    loss_axis.set_xlabel("Episode")
    loss_axis.set_ylabel("Mean squared Bellman-target error")
    loss_axis.grid(True, alpha=0.3)

    epsilon_axis = axes[1, 0]
    epsilon_axis.plot(
        episodes,
        agent.epsilon_history,
        color="green",
        linewidth=2,
    )
    epsilon_axis.set_title("Exploration Rate")
    epsilon_axis.set_xlabel("Episode")
    epsilon_axis.set_ylabel("Epsilon")
    epsilon_axis.grid(True, alpha=0.3)

    summary_axis = axes[1, 1]
    summary_text = (
        "Evaluation summary\n\n"
        f"Training episodes: {len(episode_rewards)}\n"
        f"Average evaluation reward: {avg_reward:.3f}\n"
        f"Average waiting time: {avg_metrics['avg_waiting']:.2f} min\n"
        f"Throughput: {avg_metrics['throughput']:.1f} trucks/hour\n"
        f"Gate utilization: {avg_metrics['utilization'] * 100:.1f}%\n\n"
        "DQN settings\n"
        f"Learning rate: {agent.lr}\n"
        f"Discount factor: {agent.gamma}\n"
        f"Replay batch size: {agent.batch_size}\n"
        "Hidden layers: 64 and 32 units\n"
        f"Final epsilon: {agent.epsilon:.3f}"
    )
    summary_axis.text(
        0.05,
        0.95,
        summary_text,
        fontsize=10,
        verticalalignment="top",
        bbox={"boxstyle": "round", "facecolor": "lightgreen", "alpha": 0.5},
    )
    summary_axis.set_title("Run Summary")
    summary_axis.axis("off")

    plt.tight_layout()
    plt.show()

    print("\nTraining and evaluation diagnostics")
    print("The reward curve summarizes the learning signal across episodes.")
    print("The loss curve tracks the fitted Bellman targets, not waiting time directly.")
    print("The epsilon curve shows the exploration rate recorded during training.")
    print("Evaluation metrics describe the sampled evaluation episodes.")
else:
    print("No training episodes are available to plot.")

## Interpreting the experiment

The experiment demonstrates how a DQN can learn a gate-selection policy through interaction with a simulated system. The neural network approximates action values; replay reuses past transitions; and the target network provides a more slowly changing reference for the Bellman updates. The measured reward, waiting time, throughput, and utilization describe different aspects of performance and should be considered together.

A rising or stable reward curve can be consistent with learning, but does not by itself establish convergence, optimality, or superiority to another policy. Evaluation on a small number of randomly generated episodes is subject to sampling variation. A claim about comparative performance would require a defined baseline and a controlled evaluation using comparable scenarios; this notebook does not implement such a comparison.

### Strengths and limitations
- **Potential strength:** the policy is learned from state-and-reward feedback rather than being specified as a fixed gate-selection rule.
- **Potential strength:** the neural network can represent nonlinear relationships between the encoded state and estimated action values.
- **Limitation:** training requires repeated interaction and its outcome can depend on random experiences and hyperparameters.
- **Limitation:** Q-values do not directly explain why an action was selected, and a learned policy is not guaranteed to generalize to different operating conditions.
- **Limitation:** the simulated state and reward omit operational details such as gate compatibility, truck travel, and gate-specific service rates. A policy learned under these assumptions should not be treated as an operational recommendation without a suitable model and validation.

### When this approach may be useful
A learned policy may be worth investigating when decisions are sequential, the state contains useful information about system conditions, and a representative simulator or safe source of experience is available. Simpler, transparent decision rules may be preferable when the system is stable, data are limited, or decisions must be readily explained. Further work can extend the same environment with additional operational constraints and evaluate the learned policy across more episodes; those extensions are outside the scope of this implementation.